In [1]:
import os, sys, re, time, h5py
import numpy as np
from lib import NanonisAPI, MLAAPI
from lib import FileFunctions, DataProcessing
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import sklearn.gaussian_process as gp
from sklearn.model_selection import train_test_split
from scipy.signal import fftconvolve

data = DataProcessing()
file_functions = FileFunctions()
(hw_config, error) = file_functions.load_yaml("./sys/config.yml")

#nanonis = NanonisAPI(hw_config = hw_config, message_callback = lambda message, message_type: print(f"{message} [{message_type}]"), status_callback = lambda status_str: print(f"Status: {status_str}"))
#mla = MLAAPI(hw_config = hw_config, message_callback = lambda message, message_type: print(f"{message} [{message_type}]"), status_callback = lambda status_str: print(f"Status: {status_str}"))
#nhw = nanonis.nanonis_hardware
#nanonis.link()

In [3]:
folder_path = "C:\\Expts\\Testdata\\RandomSTMdata"
file_name = "MLA_mapping_024.hdf5"
file_path = os.path.join(folder_path, file_name)

file_data = file_functions.read_hdf5(file_path)

In [29]:
data_array = file_data.get("array")

new_data_array = np.vstack([data_array[index] + 1j * data_array[index + 1] for index in np.arange(6, 70, 2)], dtype = np.complex64)
new_data_array[0]

array([0.6368491 +0.79366773j, 0.6356276 +0.79359484j,
       0.6371886 +0.79456735j, 0.6373843 +0.7950204j ,
       0.63713   +0.7929886j , 0.63763773+0.79379535j,
       0.6370195 +0.79303956j, 0.6376519 +0.7939233j ,
       0.6375676 +0.79302937j, 0.63607633+0.7931601j ,
       0.6351169 +0.7959785j , 0.6351717 +0.79518694j,
       0.6385954 +0.7928526j , 0.63603467+0.7938367j ,
       0.63500017+0.7939915j , 0.63451713+0.7935249j ,
       0.6365534 +0.79291046j, 0.6368743 +0.7928754j ,
       0.6354079 +0.7924645j , 0.6352059 +0.79283696j,
       0.63533324+0.7951663j , 0.63685507+0.7956497j ,
       0.6385917 +0.79427934j, 0.63598275+0.7932861j ,
       0.6381397 +0.7952779j , 0.63619214+0.7936148j ,
       0.6373172 +0.79369926j, 0.6355694 +0.7943037j ,
       0.63666046+0.7944367j , 0.6391191 +0.7947613j ,
       0.63690376+0.7926914j , 0.63502306+0.7930909j ,
       0.6350401 +0.79471403j, 0.63887864+0.79452354j,
       0.6364369 +0.7948386j , 0.6373429 +0.79375017j,
       0.6

In [ ]:
z_imgs = [scans[i][0, 2] for i in range(2)]
scan_range_nm = grids[0].get("scan_range (nm)")
[w_nm, h_nm] = scan_range_nm
[pix, lines] = z_imgs[0].shape
width_per_pix_nm = w_nm / pix
height_per_line_nm = h_nm / lines

imgs_norm = [z_imgs[i] - np.mean(z_imgs[i]) for i in range(2)]

img_correlation = fftconvolve(imgs_norm[0], imgs_norm[1][::-1, ::-1], mode = "same")
y_peak, x_peak = np.unravel_index(np.argmax(img_correlation), img_correlation.shape)

y_center, x_center = img_correlation.shape[0] // 2, img_correlation.shape[1] // 2
y_shift_nm = (y_peak - y_center) * height_per_line_nm
x_shift_nm = (x_peak - x_center) * width_per_pix_nm

print(f"Detected Shift (Y, X): ({y_shift_nm}, {x_shift_nm})")



fig, ax = plt.subplots(1, 3, figsize = (10, 5))
ax[0].imshow(np.hstack((z_imgs[0], z_imgs[1], img_correlation / 1000)))
ax[1].imshow(z_imgs[1])
ax[2].imshow(img_correlation)
plt.show()

In [ ]:
file_path = "C:\\Nanonis\\2026-05-29\\scan_052.hdf5"
with h5py.File(file_path, "r") as f:
    grid_attrs = f["grid"].attrs
    grid = {key: grid_attrs[key] for key in grid_attrs.keys()}
    scan = f["scan_group"]["scan"][:]
z_fwd_img = scan[0, 2]

[w_nm, h_nm] = grid["scan_range (nm)"]
[n_dirs, n_channels, pixels, lines] = scan.shape
w_pix = w_nm / pixels
h_pix = h_nm / lines

x_linspace = np.linspace(w_pix / 2 - w_nm / 2, w_nm / 2 - w_pix /2, pixels)
y_linspace = np.linspace(h_pix / 2 - h_nm / 2, h_nm / 2 - h_pix /2, lines)
x_grid, y_grid = np.meshgrid(x_linspace, y_linspace)
x_list = x_grid.flatten()
y_list = y_grid.flatten()
list_indices = np.arange(len(x_list))
xy_all_coords_nm = np.array([[x_list[index], y_list[index]] for index in list_indices])

train_indices, test_indices = train_test_split(list_indices, test_size = .999)
grid_train_indices = np.array([[index % pixels, int(index / pixels)] for index in train_indices])
xy_train_coords_nm = np.array([[x_grid[pixel, line], y_grid[pixel, line]] for pixel, line in grid_train_indices])
z_train_coords_nm = np.array([z_fwd_img[pixel, line] for pixel, line in grid_train_indices])

k_fine_detail = gp.kernels.Matern(length_scale = .3, length_scale_bounds = (.1, 1.0), nu = 1.5)
#k_intermediate = gp.kernels.Matern(length_scale = 4, length_scale_bounds = (1.5, 8.0), nu = 1.5)
k_large_scale = gp.kernels.Matern(length_scale = 20, length_scale_bounds = (5, 100), nu = 2.5)
white_kernel = gp.kernels.WhiteKernel(noise_level = .01, noise_level_bounds = (1e-6, .2))
composite_kernel = k_fine_detail + k_large_scale

print("fitting")
gpr = gp.GaussianProcessRegressor(composite_kernel, normalize_y = True)
gpr.fit(xy_train_coords_nm, z_train_coords_nm)

print("extrapolating")
(z_fit_col, z_std_dev_col) = gpr.predict(xy_all_coords_nm, return_std = True)
z_fit_nm = z_fit_col.reshape(x_grid.shape)
z_std_dev_nm = z_std_dev_col.reshape(x_grid.shape)

print("plotting")
fig, ax = plt.subplots(1, 3)
ax[0].imshow(z_fwd_img)
ax[1].imshow(z_fit_nm)
ax[2].imshow(z_std_dev_nm)
plt.show()

In [ ]:
z_std_dev_nm_masked = z_std_dev_col.copy()

fine_length_scale2 = (gpr.kernel_.get_params().get("k1__length_scale", .5)) ** 2
selected_indices = []

for point in range(10):
    best_index = np.argmax(z_std_dev_nm_masked)
    selected_indices.append(best_index)
    next_coord = xy_all_coords_nm[best_index]
    
    # Mask the region of the selected point
    for index, xy in enumerate(xy_all_coords_nm):
        if np.sum((xy - next_coord) ** 2, axis = -1) < fine_length_scale2: z_std_dev_nm_masked[index] = 0

z_std_dev_masked = z_std_dev_nm_masked.reshape(x_grid.shape)
plt.imshow(z_std_dev_masked)
plt.plot()


In [ ]:
x_list = np.linspace(-10, 10, 101, dtype = np.float32)
mask = np.ones_like(x_list, dtype = np.float32)

for index, x in enumerate(x_list):
    mask[index] *= 1 - 1 / (1 + np.sum((x - 1.5) ** 2, axis = -1))

plt.plot(x_list, mask)
plt.show()

In [19]:
xy_coords = np.column_stack((x.reshape(-1), y.reshape(-1)))
z_col = z.reshape(-1, 1)

guess_l = (2., 1.)  # In general, x and y have different scales
bounds_l = ((1e-1, 100.),) * 2  # Same bounds for x and y
guess_n = 1.  # Amount of noise
bounds_n = (1e-20, 10.) # Bounds for noise
kernel = gp.kernels.RBF()

xy_train, xy_test, z_train, z_test = train_test_split(xy_coords, z_col, test_size = 0.95)

gpr = gp.GaussianProcessRegressor(kernel, normalize_y = True)


In [26]:
xy_coords.shape

(400, 2)

In [22]:
(z_fit_col, z_std_dev_col) = gpr.predict(xy_coords, return_std = True)
z_fit = z_fit_col.reshape(x.shape)
z_std_dev = z_std_dev_col.reshape(x.shape)

In [2]:
(grid, error) = nanonis.grid_update()
(lists, error) = nanonis.grids_to_lists(grid, direction = "down")

[x_list, y_list] = [lists.get(key) for key in ["x_list (nm)", "y_list (nm)"]]
list_len = len(x_list)

rng = np.random.default_rng()
random_flat_indices = rng.choice(a = list_len, size = 20, replace = False)
random_coordinates = np.array([[x_list[index], y_list[index]] for index in random_flat_indices])

Status: running
